In [14]:
# PART A — Install everything
!pip install -q -U \
    "transformers>=4.51.0" \
    "peft>=0.16.0" \
    "trl>=0.17.0" \
    "accelerate>=1.2.0" \
    "datasets>=3.0.0" \
    "bitsandbytes>=0.45.0" \
    "torchao>=0.16.0" \
    scikit-learn \
    pandas==2.2.3




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 34.1 MB/s eta 0:00:00


In [15]:
#check GPU
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [16]:
#PART C — Imports
import json
import time
import re
import gc
import os

import torch
import pandas as pd

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments
)

from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel,
    prepare_model_for_kbit_training
)

from trl import SFTTrainer

In [17]:
#PART D — Select Qwen
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
#MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

#Create training data
#Now create examples representing the company's policy.

training_examples = [
    {
        "input": "My card was charged ₹12,500 but the order was never created.",
        "output": {
            "category": "PAYMENT",
            "sentiment": "NEGATIVE",
            "priority": "P1",
            "action": "ESCALATE_PAYMENT_TEAM"
        }
    },
    {
        "input": "₹8,999 was deducted from my account but checkout failed.",
        "output": {
            "category": "PAYMENT",
            "sentiment": "NEGATIVE",
            "priority": "P1",
            "action": "ESCALATE_PAYMENT_TEAM"
        }
    },
    {
        "input": "Payment succeeded but I did not receive an order confirmation.",
        "output": {
            "category": "PAYMENT",
            "sentiment": "NEGATIVE",
            "priority": "P1",
            "action": "ESCALATE_PAYMENT_TEAM"
        }
    },
    {
        "input": "My package was supposed to arrive yesterday but it hasn't arrived.",
        "output": {
            "category": "DELIVERY",
            "sentiment": "NEGATIVE",
            "priority": "P2",
            "action": "ROUTE_LOGISTICS"
        }
    },
    {
        "input": "Tracking has not changed for three days and my order is late.",
        "output": {
            "category": "DELIVERY",
            "sentiment": "NEGATIVE",
            "priority": "P2",
            "action": "ROUTE_LOGISTICS"
        }
    },
    {
        "input": "My parcel is delayed by four days. Please check.",
        "output": {
            "category": "DELIVERY",
            "sentiment": "NEGATIVE",
            "priority": "P2",
            "action": "ROUTE_LOGISTICS"
        }
    },
    {
        "input": "The laptop arrived with a cracked screen.",
        "output": {
            "category": "PRODUCT_DEFECT",
            "sentiment": "NEGATIVE",
            "priority": "P2",
            "action": "INITIATE_REPLACEMENT"
        }
    },
    {
        "input": "The headphones I received have no sound in the left speaker.",
        "output": {
            "category": "PRODUCT_DEFECT",
            "sentiment": "NEGATIVE",
            "priority": "P2",
            "action": "INITIATE_REPLACEMENT"
        }
    },
    {
        "input": "The mixer stopped working on the first day.",
        "output": {
            "category": "PRODUCT_DEFECT",
            "sentiment": "NEGATIVE",
            "priority": "P2",
            "action": "INITIATE_REPLACEMENT"
        }
    },
    {
        "input": "I returned my order last week but still haven't received my refund.",
        "output": {
            "category": "REFUND",
            "sentiment": "NEGATIVE",
            "priority": "P1",
            "action": "ESCALATE_REFUND_TEAM"
        }
    },
    {
        "input": "My refund was promised five days ago but nothing has arrived.",
        "output": {
            "category": "REFUND",
            "sentiment": "NEGATIVE",
            "priority": "P1",
            "action": "ESCALATE_REFUND_TEAM"
        }
    },
    {
        "input": "The refund is still pending after cancellation.",
        "output": {
            "category": "REFUND",
            "sentiment": "NEGATIVE",
            "priority": "P1",
            "action": "ESCALATE_REFUND_TEAM"
        }
    },
    {
        "input": "Please cancel my order. I placed it by mistake.",
        "output": {
            "category": "CANCELLATION",
            "sentiment": "NEUTRAL",
            "priority": "P3",
            "action": "PROCESS_CANCELLATION"
        }
    },
    {
        "input": "I no longer need this product. Please cancel the order.",
        "output": {
            "category": "CANCELLATION",
            "sentiment": "NEUTRAL",
            "priority": "P3",
            "action": "PROCESS_CANCELLATION"
        }
    },
    {
        "input": "Cancel my purchase before it is shipped.",
        "output": {
            "category": "CANCELLATION",
            "sentiment": "NEUTRAL",
            "priority": "P3",
            "action": "PROCESS_CANCELLATION"
        }
    }
]

#Create test data
test_examples = [
    {
        "input": "₹21,000 was taken from my card but no order appears in my account.",
        "category": "PAYMENT",
        "sentiment": "NEGATIVE",
        "priority": "P1",
        "action": "ESCALATE_PAYMENT_TEAM"
    },
    {
        "input": "My order should have arrived two days ago and tracking is stuck.",
        "category": "DELIVERY",
        "sentiment": "NEGATIVE",
        "priority": "P2",
        "action": "ROUTE_LOGISTICS"
    },
    {
        "input": "The television I received has a broken display.",
        "category": "PRODUCT_DEFECT",
        "sentiment": "NEGATIVE",
        "priority": "P2",
        "action": "INITIATE_REPLACEMENT"
    },
    {
        "input": "I returned the shoes six days ago but my refund has not arrived.",
        "category": "REFUND",
        "sentiment": "NEGATIVE",
        "priority": "P1",
        "action": "ESCALATE_REFUND_TEAM"
    },
    {
        "input": "Please cancel this order because I selected the wrong product.",
        "category": "CANCELLATION",
        "sentiment": "NEUTRAL",
        "priority": "P3",
        "action": "PROCESS_CANCELLATION"
    }
]

#Create our instruction
SYSTEM_PROMPT = """
You are an AI customer-support ticket classification system.

Analyze the customer message and return ONLY valid JSON.

The JSON must contain exactly these fields:

category
sentiment
priority
action

Allowed category values:
PAYMENT
DELIVERY
PRODUCT_DEFECT
REFUND
CANCELLATION

Allowed sentiment values:
POSITIVE
NEUTRAL
NEGATIVE

Allowed priority values:
P1
P2
P3

Allowed actions:
ESCALATE_PAYMENT_TEAM
ROUTE_LOGISTICS
INITIATE_REPLACEMENT
ESCALATE_REFUND_TEAM
PROCESS_CANCELLATION
"""

#Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

#Format fine-tuning dataset
def format_training_example(example):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": example["input"]
        },
        {
            "role": "assistant",
            "content": json.dumps(example["output"])
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {
        "text": text
    }


#Create the Hugging Face dataset:
train_dataset = Dataset.from_list(training_examples)

train_dataset = train_dataset.map(
    format_training_example
)

print(train_dataset[0]["text"])



Map:   0%|          | 0/15 [00:00<?, ? examples/s]

<|im_start|>system

You are an AI customer-support ticket classification system.

Analyze the customer message and return ONLY valid JSON.

The JSON must contain exactly these fields:

category
sentiment
priority
action

Allowed category values:
PAYMENT
DELIVERY
PRODUCT_DEFECT
REFUND
CANCELLATION

Allowed sentiment values:
POSITIVE
NEUTRAL
NEGATIVE

Allowed priority values:
P1
P2
P3

Allowed actions:
ESCALATE_PAYMENT_TEAM
ROUTE_LOGISTICS
INITIATE_REPLACEMENT
ESCALATE_REFUND_TEAM
PROCESS_CANCELLATION
<|im_end|>
<|im_start|>user
My card was charged ₹12,500 but the order was never created.<|im_end|>
<|im_start|>assistant
{"category": "PAYMENT", "sentiment": "NEGATIVE", "priority": "P1", "action": "ESCALATE_PAYMENT_TEAM"}<|im_end|>



MODEL 1 — BASE QWEN

In [18]:
#Load Base Qwen

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

base_model.eval()
#Universal inference function

def generate_response(model, customer_message, max_new_tokens=120):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": customer_message
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    start_time = time.perf_counter()

    with torch.no_grad():

        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    elapsed = time.perf_counter() - start_time

    generated = output[
        0,
        inputs["input_ids"].shape[1]:
    ]

    response = tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

    return response.strip(), elapsed

#Test Base Qwen
sample_ticket = """
₹21,000 was deducted from my card,
but my order was not confirmed.
"""

response, latency = generate_response(
    base_model,
    sample_ticket
)

print(response)
print("\nLatency:", latency)


#The base model understands the complaint but does not necessarily know our internal priority policy.
#That's the purpose of fine-tuning.



Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

{
  "category": "DELIVERY",
  "sentiment": "NEGATIVE",
  "priority": "P3",
  "action": "ESCALATE_DELIVERY_TEAM"
}

Latency: 2.2353246109998963


In [19]:
#Before Training: Record Base Results
base_results = []

for item in test_examples:

    output, latency = generate_response(
        base_model,
        item["input"]
    )

    base_results.append({
        "input": item["input"],
        "expected_category": item["category"],
        "expected_sentiment": item["sentiment"],
        "expected_priority": item["priority"],
        "expected_action": item["action"],
        "output": output,
        "latency": latency
    })

In [20]:
#Free Base Model Memory
del base_model

gc.collect()

torch.cuda.empty_cache()

MODEL 2 — QWEN + LoRA

What LoRA does

Normal fine-tuning modifies billions of parameters.

LoRA freezes Qwen and adds small matrices:

In [21]:
# ============================================================
# MODEL 2: QWEN + LoRA
# ============================================================

import torch
import gc
import time

from transformers import (
    AutoModelForCausalLM,
    TrainingArguments
)

from peft import (
    LoraConfig,
    get_peft_model
)

from trl import SFTTrainer


# ------------------------------------------------------------
# 1. Clean GPU memory before loading LoRA model
# ------------------------------------------------------------

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


# ------------------------------------------------------------
# 2. Check GPU
# ------------------------------------------------------------

use_bf16 = (
    torch.cuda.is_available()
    and torch.cuda.is_bf16_supported()
)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("BF16 supported:", use_bf16)


# ------------------------------------------------------------
# 3. Load fresh Qwen base model for LoRA
# ------------------------------------------------------------
# IMPORTANT:
# Do NOT load the model directly as FP16.
#
# BF16 supported GPU -> BF16
# Otherwise -> FP32 model + AMP during training
# ------------------------------------------------------------

model_dtype = (
    torch.bfloat16
    if use_bf16
    else torch.float32
)

print("Loading LoRA base model as:", model_dtype)


lora_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=model_dtype,
    device_map="auto"
)

lora_model.config.use_cache = False


# ------------------------------------------------------------
# 4. Configure LoRA
# ------------------------------------------------------------

lora_config = LoraConfig(

    r=16,

    lora_alpha=32,

    lora_dropout=0.05,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],

    bias="none",

    task_type="CAUSAL_LM"
)


# ------------------------------------------------------------
# 5. Attach LoRA adapters
# ------------------------------------------------------------

lora_model = get_peft_model(
    lora_model,
    lora_config
)


print("\nTrainable Parameters:")

lora_model.print_trainable_parameters()

CUDA available: True
GPU: Tesla T4
BF16 supported: True
Loading LoRA base model as: torch.bfloat16


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


Trainable Parameters:
trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


In [22]:
# ------------------------------------------------------------
# 6. Configure LoRA Training
# ------------------------------------------------------------

lora_training_args = TrainingArguments(

    output_dir="./qwen_lora",

    num_train_epochs=5,

    # 1 is safer for Colab GPUs
    per_device_train_batch_size=1,

    # maintains effective batch size
    gradient_accumulation_steps=8,

    learning_rate=2e-4,

    logging_steps=1,

    save_strategy="no",

    report_to="none",

    # Use BF16 only when GPU supports it
    bf16=use_bf16,

    # T4 etc:
    # model is FP32 but computation uses FP16 AMP
    fp16=(
        torch.cuda.is_available()
        and not use_bf16
    ),

    optim="adamw_torch",

    remove_unused_columns=False
)

# ------------------------------------------------------------
# 7. Create LoRA Trainer
# ------------------------------------------------------------

lora_trainer = SFTTrainer(
    model=lora_model,
    train_dataset=train_dataset,
    args=lora_training_args
)



# ------------------------------------------------------------
# 8. Train LoRA
# ------------------------------------------------------------

print("\nStarting LoRA training...\n")

start_time = time.perf_counter()

train_result = lora_trainer.train()

lora_train_time = (
    time.perf_counter()
    - start_time
)

print("\nLoRA training completed successfully.")

print(
    f"Training time: {lora_train_time:.2f} seconds"
)

# ------------------------------------------------------------
# 9. Save LoRA Adapter
# ------------------------------------------------------------

lora_model.save_pretrained(
    "./qwen_lora_adapter"
)

tokenizer.save_pretrained(
    "./qwen_lora_adapter"
)

print("LoRA adapter saved.")


# ------------------------------------------------------------
# 10. Test LoRA
# ------------------------------------------------------------

lora_model.eval()

response, latency = generate_response(
    lora_model,
    sample_ticket
)

print("LoRA Response:")
print(response)

print(
    f"\nInference latency: {latency:.3f} sec"
)


# ------------------------------------------------------------
# 11. Evaluate LoRA
# ------------------------------------------------------------

lora_results = []

lora_model.eval()


for item in test_examples:

    output, latency = generate_response(
        lora_model,
        item["input"]
    )

    lora_results.append({

        "input":
            item["input"],

        "expected_category":
            item["category"],

        "expected_sentiment":
            item["sentiment"],

        "expected_priority":
            item["priority"],

        "expected_action":
            item["action"],

        "output":
            output,

        "latency":
            latency
    })


print(
    "Number of test examples evaluated:",
    len(lora_results)
)


# ------------------------------------------------------------
# 12. Record LoRA GPU Memory
# ------------------------------------------------------------

if torch.cuda.is_available():

    lora_peak_memory = (
        torch.cuda.max_memory_allocated()
        / 1024**3
    )

else:

    lora_peak_memory = 0


print(
    f"LoRA Peak GPU Memory: "
    f"{lora_peak_memory:.2f} GB"
)


# ------------------------------------------------------------
# 13. Free LoRA Memory
# ------------------------------------------------------------

del lora_model
del lora_trainer

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("LoRA model removed from GPU.")

Adding EOS to train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.



Starting LoRA training...



Step,Training Loss
1,3.062071
2,2.651605
3,2.475230
4,2.337307
5,2.222214
6,2.114386
7,2.008823
8,1.969550
9,1.897910
10,1.881905



LoRA training completed successfully.
Training time: 48.91 seconds
LoRA adapter saved.
LoRA Response:
{
  "category": "DELIVERY",
  "sentiment": "NEGATIVE",
  "priority": "P3",
  "action": "ESCALATE_DELIVERY_TEAM"
}

Inference latency: 2.622 sec
Number of test examples evaluated: 5
LoRA Peak GPU Memory: 11.54 GB
LoRA model removed from GPU.


MODEL 3 — QWEN + QLoRA

QLoRA does not mean the LoRA adapter itself is simply "4-bit LoRA."

The important idea is that the frozen base model is quantized, dramatically reducing GPU memory during fine-tuning.


In [25]:
# ============================================================
# MODEL 3: QWEN + QLoRA
# Tesla T4 Safe Version
# ============================================================

import gc
import time
import torch

from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

from trl import SFTTrainer


# ------------------------------------------------------------
# 1. Clean GPU memory
# ------------------------------------------------------------

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


print("GPU:", torch.cuda.get_device_name(0))

GPU: Tesla T4


In [27]:
# ------------------------------------------------------------
# 2. QLoRA 4-bit Configuration
# ------------------------------------------------------------

bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    # NF4 is recommended for QLoRA
    bnb_4bit_quant_type="nf4",

    # Additional memory saving
    bnb_4bit_use_double_quant=True,

    # IMPORTANT FOR TESLA T4
    bnb_4bit_compute_dtype=torch.float16
)


# ------------------------------------------------------------
# 3. Load 4-bit Qwen
# ------------------------------------------------------------

qlora_model = AutoModelForCausalLM.from_pretrained(

    MODEL_NAME,

    quantization_config=bnb_config,

    device_map="auto"
)

qlora_model.config.use_cache = False

print("4-bit Qwen loaded.")

# ------------------------------------------------------------
# 4. Prepare model for QLoRA training
# ------------------------------------------------------------

qlora_model = prepare_model_for_kbit_training(
    qlora_model,
    use_gradient_checkpointing=True
)


# ------------------------------------------------------------
# 5. QLoRA Adapter Configuration
# ------------------------------------------------------------

qlora_config = LoraConfig(

    r=16,

    lora_alpha=32,

    lora_dropout=0.05,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],

    bias="none",

    task_type="CAUSAL_LM"
)


# ------------------------------------------------------------
# 6. Add LoRA adapters
# ------------------------------------------------------------

qlora_model = get_peft_model(
    qlora_model,
    qlora_config
)

qlora_model.print_trainable_parameters()


# ------------------------------------------------------------
# 7. Ensure trainable LoRA parameters are FP32
# ------------------------------------------------------------

for name, param in qlora_model.named_parameters():

    if param.requires_grad:

        param.data = param.data.float()


from collections import Counter

trainable_dtypes = Counter()

for name, param in qlora_model.named_parameters():
    if param.requires_grad:
        trainable_dtypes[str(param.dtype)] += param.numel()

print("Trainable parameter dtypes:")

for dtype, count in trainable_dtypes.items():
    print(dtype, ":", count)

# ------------------------------------------------------------
# 8. QLoRA Training Configuration
# ------------------------------------------------------------

qlora_training_args = TrainingArguments(

    output_dir="./qwen_qlora",

    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="no",
    report_to="none",

    # IMPORTANT:
    # Disable Trainer AMP / GradScaler
    fp16=False,

    bf16=False,

    optim="paged_adamw_8bit",
    remove_unused_columns=False,
    gradient_checkpointing=True,

    gradient_checkpointing_kwargs={
        "use_reentrant": False
    }
)


# ------------------------------------------------------------
# 9. QLoRA Trainer
# ------------------------------------------------------------

qlora_trainer = SFTTrainer(

    model=qlora_model,

    train_dataset=train_dataset,

    args=qlora_training_args
)



# ------------------------------------------------------------
# 10. Train QLoRA
# ------------------------------------------------------------

print("\nStarting QLoRA training...\n")

start = time.perf_counter()

qlora_trainer.train()

qlora_train_time = (
    time.perf_counter()
    - start
)

print("\nQLoRA training completed!")

print(
    f"Training time: "
    f"{qlora_train_time:.2f} seconds"
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

4-bit Qwen loaded.
trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815
Trainable parameter dtypes:
torch.float32 : 4358144


Adding EOS to train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

...

Dropping fully masked examples from train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.



Starting QLoRA training...



Step,Training Loss
1,3.012223
2,2.549376
3,2.368655
4,2.225565
5,2.121987
6,2.033606
7,1.937489
8,1.900397
9,1.831283
10,1.811931



QLoRA training completed!
Training time: 46.74 seconds


In [28]:
# ------------------------------------------------------------
# 11. Save QLoRA Adapter
# ------------------------------------------------------------

qlora_model.save_pretrained(
    "./qwen_qlora_adapter"
)

tokenizer.save_pretrained(
    "./qwen_qlora_adapter"
)

print("QLoRA adapter saved.")




QLoRA adapter saved.


In [29]:
qlora_model.eval()

response, latency = generate_response(
    qlora_model,
    sample_ticket
)

print("QLoRA Response:")
print(response)

print(
    f"\nLatency: {latency:.3f} seconds"
)

qlora_results = []

qlora_model.eval()


for item in test_examples:

    output, latency = generate_response(
        qlora_model,
        item["input"]
    )

    qlora_results.append({

        "input":
            item["input"],

        "expected_category":
            item["category"],

        "expected_sentiment":
            item["sentiment"],

        "expected_priority":
            item["priority"],

        "expected_action":
            item["action"],

        "output":
            output,

        "latency":
            latency
    })

QLoRA Response:
```json
{
  "category": "DELIVERY",
  "sentiment": "NEGATIVE",
  "priority": "P3",
  "action": "ESCALATE_DELIVERY"
}
```

Latency: 12.199 seconds


NOW THE IMPORTANT PART — EVALUATION

In [30]:
def extract_json(text):

    try:

        # First try direct JSON
        return json.loads(text)

    except:

        pass


    try:

        # Remove markdown fences
        cleaned = text.replace(
            "```json", ""
        ).replace(
            "```", ""
        ).strip()

        return json.loads(cleaned)

    except:

        pass


    try:

        # Find first JSON-looking object
        match = re.search(
            r'\{.*\}',
            text,
            re.DOTALL
        )

        if match:

            return json.loads(
                match.group()
            )

    except:

        pass


    return None

In [34]:
REQUIRED_FIELDS = [
    "category",
    "sentiment",
    "priority",
    "action"
]


def evaluate_model(results):

    total = len(results)
    valid_json = 0
    schema_correct = 0
    category_correct = 0
    sentiment_correct = 0
    priority_correct = 0
    action_correct = 0
    exact_match = 0
    latencies = []


    for item in results:

        latencies.append(
            item["latency"]
        )

        parsed = extract_json(
            item["output"]
        )

        if parsed is None:
            continue


        valid_json += 1


        if all(
            field in parsed
            for field in REQUIRED_FIELDS
        ):
            schema_correct += 1


        category_ok = (
            parsed.get("category")
            == item["expected_category"]
        )

        sentiment_ok = (
            parsed.get("sentiment")
            == item["expected_sentiment"]
        )

        priority_ok = (
            parsed.get("priority")
            == item["expected_priority"]
        )

        action_ok = (
            parsed.get("action")
            == item["expected_action"]
        )


        category_correct += category_ok

        sentiment_correct += sentiment_ok

        priority_correct += priority_ok

        action_correct += action_ok


        if (
            category_ok
            and sentiment_ok
            and priority_ok
            and action_ok
        ):
            exact_match += 1


    return {

        "JSON Validity %":
            100 * valid_json / total,

        "Schema Compliance %":
            100 * schema_correct / total,

        "Category Accuracy %":
            100 * category_correct / total,

        "Sentiment Accuracy %":
            100 * sentiment_correct / total,

        "Priority Accuracy %":
            100 * priority_correct / total,

        "Action Accuracy %":
            100 * action_correct / total,

        "Exact Record Accuracy %":
            100 * exact_match / total,

        "Avg Latency (sec)":
            sum(latencies) / len(latencies)
    }

In [39]:
#Evaluate all three
base_metrics = evaluate_model(
    base_results
)

lora_metrics = evaluate_model(
    lora_results
)

qlora_metrics = evaluate_model(
    qlora_results
)


# ------------------------------------------------------------
# Print individual results
# ------------------------------------------------------------

print("\n========== BASE QWEN ==========")

for metric, value in base_metrics.items():
    print(f"{metric:<28}: {value:.2f}")


print("\n========== LoRA ==========")

for metric, value in lora_metrics.items():
    print(f"{metric:<28}: {value:.2f}")


print("\n========== QLoRA ==========")

for metric, value in qlora_metrics.items():
    print(f"{metric:<28}: {value:.2f}")


========== BASE QWEN ==========
JSON Validity %             : 100.00
Schema Compliance %         : 100.00
Category Accuracy %         : 80.00
Sentiment Accuracy %        : 80.00
Priority Accuracy %         : 0.00
Action Accuracy %           : 20.00
Exact Record Accuracy %     : 0.00
Avg Latency (sec)           : 1.74

========== LoRA ==========
JSON Validity %             : 100.00
Schema Compliance %         : 100.00
Category Accuracy %         : 80.00
Sentiment Accuracy %        : 80.00
Priority Accuracy %         : 20.00
Action Accuracy %           : 60.00
Exact Record Accuracy %     : 0.00
Avg Latency (sec)           : 2.76

========== QLoRA ==========
JSON Validity %             : 100.00
Schema Compliance %         : 100.00
Category Accuracy %         : 60.00
Sentiment Accuracy %        : 80.00
Priority Accuracy %         : 0.00
Action Accuracy %           : 40.00
Exact Record Accuracy %     : 0.00
Avg Latency (sec)           : 4.20


EXPECTED

Metric	Base Qwen	LoRA	QLoRA  <br>
JSON validity	80%	100%	100%  <br>
Schema compliance	70%	100%	100%  <br>
Category accuracy	80%	95%	95%  <br>
Sentiment accuracy	90%	100%	100%  <br>
Priority accuracy	50%	95%	90%  <br>
Action accuracy	60%	95%	95%  <br>
Exact record	40%	90%	85–90%  <br>
Training memory	—	Higher	Much lower  <br>

In [40]:
# ============================================================
# FINAL EVALUATION: BASE vs LoRA vs QLoRA
# ============================================================

import pandas as pd


# ------------------------------------------------------------
# 1. Evaluate all models
# ------------------------------------------------------------

base_metrics = evaluate_model(base_results)
lora_metrics = evaluate_model(lora_results)
qlora_metrics = evaluate_model(qlora_results)


# ------------------------------------------------------------
# 2. Check that metrics were created
# ------------------------------------------------------------

print("Base metrics:")
print(base_metrics)

print("\nLoRA metrics:")
print(lora_metrics)

print("\nQLoRA metrics:")
print(qlora_metrics)


# ------------------------------------------------------------
# 3. Calculate overall business accuracy
# ------------------------------------------------------------

def overall_business_accuracy(metrics):

    required = [
        "Category Accuracy %",
        "Sentiment Accuracy %",
        "Priority Accuracy %",
        "Action Accuracy %"
    ]

    # Check required metrics exist
    missing = [
        x for x in required
        if x not in metrics
    ]

    if missing:
        print(
            "Missing metrics:",
            missing
        )
        return 0

    return sum(
        metrics[x]
        for x in required
    ) / len(required)


# ------------------------------------------------------------
# 4. Add Business Accuracy to each model
# ------------------------------------------------------------

base_metrics["Business Accuracy %"] = (
    overall_business_accuracy(base_metrics)
)

lora_metrics["Business Accuracy %"] = (
    overall_business_accuracy(lora_metrics)
)

qlora_metrics["Business Accuracy %"] = (
    overall_business_accuracy(qlora_metrics)
)


# ------------------------------------------------------------
# 5. Create comparison table
# ------------------------------------------------------------

comparison = pd.DataFrame([
    base_metrics,
    lora_metrics,
    qlora_metrics
],
index=[
    "Base Qwen",
    "LoRA",
    "QLoRA"
])


# ------------------------------------------------------------
# 6. Round values
# ------------------------------------------------------------

comparison = comparison.round(2)


# ------------------------------------------------------------
# 7. Display final comparison
# ------------------------------------------------------------

print("\n")
print("=" * 70)
print("BASE QWEN vs LoRA vs QLoRA")
print("=" * 70)

display(comparison)

Base metrics:
{'JSON Validity %': 100.0, 'Schema Compliance %': 100.0, 'Category Accuracy %': 80.0, 'Sentiment Accuracy %': 80.0, 'Priority Accuracy %': 0.0, 'Action Accuracy %': 20.0, 'Exact Record Accuracy %': 0.0, 'Avg Latency (sec)': 1.7435145397999805}

LoRA metrics:
{'JSON Validity %': 100.0, 'Schema Compliance %': 100.0, 'Category Accuracy %': 80.0, 'Sentiment Accuracy %': 80.0, 'Priority Accuracy %': 20.0, 'Action Accuracy %': 60.0, 'Exact Record Accuracy %': 0.0, 'Avg Latency (sec)': 2.763709976400105}

QLoRA metrics:
{'JSON Validity %': 100.0, 'Schema Compliance %': 100.0, 'Category Accuracy %': 60.0, 'Sentiment Accuracy %': 80.0, 'Priority Accuracy %': 0.0, 'Action Accuracy %': 40.0, 'Exact Record Accuracy %': 0.0, 'Avg Latency (sec)': 4.203637067}


BASE QWEN vs LoRA vs QLoRA


,JSON Validity %,Schema Compliance %,Category Accuracy %,Sentiment Accuracy %,Priority Accuracy %,Action Accuracy %,Exact Record Accuracy %,Avg Latency (sec),Business Accuracy %
Base Qwen,100.0,100.0,80.0,80.0,0.0,20.0,0.0,1.74,45.0
LoRA,100.0,100.0,80.0,80.0,20.0,60.0,0.0,2.76,60.0
QLoRA,100.0,100.0,60.0,80.0,0.0,40.0,0.0,4.20,45.0
